# 02 · Performance Forecasting — Gradient Boosting

**Goal** — forecast an employee's upcoming **performance score** (continuous 40–100) from
historical KPI completion (`kpi_achievement_percent`, `performance_last_year`,
`performance_two_years_ago`), attendance consistency (`attendance_rate`, `late_days`,
`avg_monthly_hours`) and training participation (`training_hours_last_year`,
`certifications_count`, `skill_assessment_score`). This gives HR an early read on likely
high/low performers, complementing the manual weighted-KPI scoring in the Performance module.

**Algorithm — Gradient Boosting** (`HistGradientBoostingRegressor`, scikit-learn's
histogram-based gradient boosting — the scalable variant for this ~100k-row table). Chosen
for its accuracy on tabular data with non-linear feature interactions. Rationale + citations
in `../MODEL-JUSTIFICATION.md`.

As confirmed, the performance model is built on the **promotion dataset**
(`employee_promotion_prediction.csv`) — its `performance_score` column is the richest
performance signal across the datasets. Logged to `logs/performance*.log`; artifacts under
`artifacts/performance/`.

> **Leakage note:** the `promoted` outcome is dropped (downstream of performance);
> historical performance columns are kept as legitimate predictors.

In [ ]:
# --- environment & logged run -------------------------------------------------------
import sys
from pathlib import Path

# notebooks/ live one level below the model root where synapse_ml.py sits
HERE = Path.cwd()
MODEL_DIR = HERE if (HERE / "synapse_ml.py").exists() else HERE.parent
sys.path.insert(0, str(MODEL_DIR))

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 80)

import synapse_ml as sm
run = sm.start_run("performance")   # opens logs/performance_<ts>.log + artifacts/performance/
log = run.log

## 1 · Load data

In [ ]:
df = run.load_csv("employee_promotion_prediction.csv")
print(df.shape)
df.head()

In [ ]:
import io
_info = io.StringIO()
df.info(buf=_info)
log.info("dataframe info:\n%s", _info.getvalue())
df.describe().T

## 2 · Target distribution & drivers

In [ ]:
TARGET = "performance_score"
ID_COL = "employee_id"
DROP = [ID_COL, "promoted"]   # promoted is downstream of performance -> leakage

y = df[TARGET].astype(float)
log.info("target=%s · mean=%.2f std=%.2f min=%.2f max=%.2f",
         TARGET, y.mean(), y.std(), y.min(), y.max())

fig, ax = plt.subplots(figsize=(6, 4))
sns.histplot(y, bins=40, kde=True, ax=ax, color="#4c72b0")
ax.set_title(f"Distribution of {TARGET}")
run.save_fig(fig, "01_target_distribution"); plt.show()

In [ ]:
num_df = df.drop(columns=DROP).select_dtypes("number")
corr = num_df.corr()[TARGET].drop(TARGET).sort_values()
log.info("numeric correlation with %s:\n%s", TARGET, corr.to_string())

fig, ax = plt.subplots(figsize=(7, 8))
corr.plot(kind="barh", ax=ax, color=np.where(corr > 0, "#dd8452", "#4c72b0"))
ax.set_title(f"Correlation of numeric features with {TARGET}")
run.save_fig(fig, "02_feature_correlation"); plt.show()

## 3 · Preprocessing & split

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

X = df.drop(columns=DROP + [TARGET])
numeric_cols, categorical_cols = sm.split_feature_types(df.drop(columns=DROP), target=TARGET)
log.info("numeric=%d · categorical=%d %s", len(numeric_cols), len(categorical_cols), categorical_cols)

preprocess = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                      ("scale", StandardScaler())]), numeric_cols),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                      ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]),
     categorical_cols),
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
log.info("split · train=%s test=%s", X_train.shape, X_test.shape)

## 4 · Gradient Boosting model

`HistGradientBoostingRegressor` — additive trees fit on the gradient of the squared-error
loss. Read by 3-fold cross-validated R² (3 folds keep the 100k-row run quick), then fit on
all training data.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import KFold

cv = KFold(n_splits=3, shuffle=True, random_state=42)
model = Pipeline([
    ("prep", preprocess),
    ("reg", HistGradientBoostingRegressor(
        learning_rate=0.05, max_iter=600, max_leaf_nodes=31,
        l2_regularization=1.0, early_stopping=True, random_state=42)),
])

cv_r2 = cross_val_score(model, X_train, y_train, cv=cv, scoring="r2", n_jobs=-1)
log.info("CV R2 = %.4f (+/- %.4f)", cv_r2.mean(), cv_r2.std())
model.fit(X_train, y_train)
log.info("fitted GradientBoosting on %d rows", len(X_train))

## 5 · Evaluation

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pred = model.predict(X_test)
mae = mean_absolute_error(y_test, pred)
rmse = float(np.sqrt(mean_squared_error(y_test, pred)))
r2 = r2_score(y_test, pred)
log.info("test · MAE=%.3f RMSE=%.3f R2=%.4f", mae, rmse, r2)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(y_test, pred, s=6, alpha=0.25, color="#4c72b0")
lims = [y_test.min(), y_test.max()]; axes[0].plot(lims, lims, "k--", lw=1)
axes[0].set_xlabel("actual"); axes[0].set_ylabel("predicted")
axes[0].set_title(f"Predicted vs actual · R2={r2:.3f}")
resid = y_test - pred
axes[1].scatter(pred, resid, s=6, alpha=0.25, color="#c44e52"); axes[1].axhline(0, color="k", lw=1)
axes[1].set_xlabel("predicted"); axes[1].set_ylabel("residual")
axes[1].set_title(f"Residuals · MAE={mae:.2f} RMSE={rmse:.2f}")
fig.tight_layout(); run.save_fig(fig, "03_pred_vs_actual_residuals"); plt.show()

In [ ]:
# Permutation importance (model-agnostic) on a test subsample.
from sklearn.inspection import permutation_importance

sample = X_test.sample(min(4000, len(X_test)), random_state=42)
perm = permutation_importance(model, sample, y_test.loc[sample.index], scoring="r2",
                              n_repeats=5, random_state=42, n_jobs=-1)
imp = pd.Series(perm.importances_mean, index=X_test.columns).sort_values().tail(15)
log.info("top permutation importances:\n%s", imp.sort_values(ascending=False).to_string())

fig, ax = plt.subplots(figsize=(7, 6))
imp.plot(kind="barh", ax=ax, color="#55a868")
ax.set_title("Permutation importance (R2 drop) · top 15")
run.save_fig(fig, "04_permutation_importance"); plt.show()

## 6 · Forecast roster

Forecast scores for the test set and flag the largest gaps vs. last year's score — the
employees whose trajectory is bending up or down.

In [ ]:
scored = X_test.copy()
scored["forecast_score"] = pred.round(1)
scored["actual_score"] = y_test.values.round(1)
scored["delta_vs_last_year"] = (pred - X_test["performance_last_year"]).round(1)
log.info("biggest forecast drops vs last year:\n%s",
         scored.nsmallest(5, "delta_vs_last_year")[
             ["performance_last_year", "forecast_score", "delta_vs_last_year"]].to_string())
run.checkpoint_df(scored.reset_index(names="row_id"), "forecast_roster")
scored[["performance_last_year", "forecast_score", "actual_score", "delta_vs_last_year"]].head()

## 7 · Persist model & metrics

In [ ]:
run.save_model(model, "performance_model")
run.save_metrics({
    "algorithm": "HistGradientBoostingRegressor",
    "cv_r2": float(cv_r2.mean()),
    "test_mae": mae, "test_rmse": rmse, "test_r2": r2,
    "target_mean": float(y.mean()), "target_std": float(y.std()),
    "n_train": int(len(X_train)), "n_test": int(len(X_test)),
})
run.finish(summary=f"GradientBoosting: test R2={r2:.3f}, MAE={mae:.2f}, RMSE={rmse:.2f}")